# 面试问题：多模态文档 RAG 怎样检索页面、版面区域、表格和图片，同时保留可核验引用？

**一句话回答。** 摄取时将页面拆成带 bbox、模态、OCR/表格质量、原始文件版本和 ACL 的 region；查询按文本/表格/图片需求路由并以 region 为最小引用单位。不要把 OCR 文本、表格单元格和图像说明扁平拼接后丢掉页码与坐标。

本 Notebook 以小型、受控数据实现必要的数据合同、验证器和状态机。它不访问真实网站、文件或模型，也不把断言结果宣传成生产质量、安全保证或法律合规结论。

**资料入口。** [SK-VQA](https://arxiv.org/abs/2406.19593) 讨论多模态检索增强生成所需的图像与外部上下文；本例聚焦文档版面 provenance。


In [ ]:
question = "Multimodal RAG 版面检索"  # 执行本行的状态、计算或校验逻辑。
assert "RAG" in question  # 执行本行的状态、计算或校验逻辑。
assert 7 > 6  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. region 是多模态文档的最小可引用对象

同一页可能同时有正文、表格、图片和页眉。每个 region 需要页码、bbox、模态、提取文本/结构、质量分、原文件 revision 与 ACL；否则答案无法回跳到 PDF 的具体位置。


In [ ]:
regions = [{"id": "r1", "page": 1, "bbox": (0, 0, 400, 120), "kind": "text", "text": "退款需人工确认", "quality": 0.99, "doc": "policy-v2", "acl": "support"}, {"id": "r2", "page": 1, "bbox": (0, 130, 400, 260), "kind": "table", "text": "状态|到账天数;退款|3", "quality": 0.97, "doc": "policy-v2", "acl": "support"}, {"id": "r3", "page": 2, "bbox": (0, 0, 400, 300), "kind": "image", "text": "流程图显示确认后到账", "quality": 0.88, "doc": "policy-v2", "acl": "support"}]  # 执行本行的状态、计算或校验逻辑。
assert len(regions) == 3  # 执行本行的状态、计算或校验逻辑。
assert {region["kind"] for region in regions} == {"text", "table", "image"}  # 执行本行的状态、计算或校验逻辑。
assert all(region["doc"] == "policy-v2" for region in regions)  # 执行本行的状态、计算或校验逻辑。

## 2. 查询路由决定优先检索的模态，但不丢其他证据

问“到账天数”时优先 table，问流程时可取 text/image。路由器可以是规则或模型，但应输出可审计意图和 fallback；错误路由不能静默导致表格事实只用 OCR 段落回答。


In [ ]:
def route(query):  # 执行本行的状态、计算或校验逻辑。
    return "table" if "天数" in query else "image" if "流程图" in query else "text"  # 执行本行的状态、计算或校验逻辑。
assert route("退款到账天数") == "table"  # 执行本行的状态、计算或校验逻辑。
assert route("查看流程图") == "image"  # 执行本行的状态、计算或校验逻辑。
assert route("退款条件") == "text"  # 执行本行的状态、计算或校验逻辑。

## 3. region ranking 保留模态与 ACL 过滤

教学按字符重叠排序，并对目标模态加分。生产可以用文本 embedding、图像 embedding、layout model 和 cross-encoder，但所有候选仍必须在权限过滤后带 page/bbox 返回。


In [ ]:
def score(query, region, preferred):  # 执行本行的状态、计算或校验逻辑。
    return len(set(query) & set(region["text"])) + int(region["kind"] == preferred)  # 执行本行的状态、计算或校验逻辑。
def retrieve(query, role):  # 执行本行的状态、计算或校验逻辑。
    preferred = route(query)  # 执行本行的状态、计算或校验逻辑。
    allowed = [region for region in regions if region["acl"] == role]  # 执行本行的状态、计算或校验逻辑。
    return sorted(allowed, key=lambda region: score(query, region, preferred), reverse=True)  # 执行本行的状态、计算或校验逻辑。
hits = retrieve("退款到账天数", "support")  # 执行本行的状态、计算或校验逻辑。
assert hits[0]["id"] == "r2"  # 执行本行的状态、计算或校验逻辑。
assert all(region["acl"] == "support" for region in hits)  # 执行本行的状态、计算或校验逻辑。
assert all(region["doc"] == "policy-v2" for region in hits)  # 执行本行的状态、计算或校验逻辑。

## 4. 表格必须保留单元格语义，而非只留一串 OCR

表格检索命中后应保存行/列/表头和原页坐标。这里用简化分隔符解析；真实表格会有合并单元格、跨页、脚注和单位，需在结构提取质量不足时标注不确定并回看图像。


In [ ]:
def parse_table(region):  # 执行本行的状态、计算或校验逻辑。
    header, row = region["text"].split(";")  # 执行本行的状态、计算或校验逻辑。
    columns = header.split("|")  # 执行本行的状态、计算或校验逻辑。
    values = row.split("|")  # 执行本行的状态、计算或校验逻辑。
    return dict(zip(columns, values))  # 执行本行的状态、计算或校验逻辑。
table = parse_table(hits[0])  # 执行本行的状态、计算或校验逻辑。
assert table == {"状态": "退款", "到账天数": "3"}  # 执行本行的状态、计算或校验逻辑。
assert table["到账天数"] == "3"  # 执行本行的状态、计算或校验逻辑。
assert hits[0]["kind"] == "table"  # 执行本行的状态、计算或校验逻辑。

## 5. 低质量 OCR/视觉提取触发回退，而非装作精确

quality 只是一个信号，不是事实正确概率。低于阈值的 region 可以要求 VLM 复读、使用原图、人工校验或拒答；任何修复输出应生成新的 extraction version，不覆盖原始 artifact。


In [ ]:
def usable(region, threshold):  # 执行本行的状态、计算或校验逻辑。
    return region["quality"] >= threshold  # 执行本行的状态、计算或校验逻辑。
assert usable(regions[0], 0.95)  # 执行本行的状态、计算或校验逻辑。
assert not usable(regions[2], 0.95)  # 执行本行的状态、计算或校验逻辑。
assert usable(regions[2], 0.80)  # 执行本行的状态、计算或校验逻辑。

## 6. 上下文装配以 region 预算为单位

不能因一个表格命中就把整份 PDF 塞进 prompt。按 region 选择、去重和 token/视觉预算，保留同页相邻 caption 或表头等必要上下文；超预算应报告被截断的 region，而不是隐藏选择偏差。


In [ ]:
def assemble(selected, budget):  # 执行本行的状态、计算或校验逻辑。
    return selected[:budget]  # 执行本行的状态、计算或校验逻辑。
context = assemble(hits, 2)  # 执行本行的状态、计算或校验逻辑。
assert [region["id"] for region in context] == ["r2", "r1"]  # 执行本行的状态、计算或校验逻辑。
assert len(context) == 2  # 执行本行的状态、计算或校验逻辑。
assert all(region["page"] == 1 for region in context)  # 执行本行的状态、计算或校验逻辑。

## 7. 引用需要文件、页码、bbox、模态和版本

用户需要点击/高亮到具体区域，评测也需要判断答案是否真的由表格或图片支持。引用只给文档名或 chunk id 不能解释跨页表格和 OCR 误读；bbox 是 UI 与审计的共同接口。


In [ ]:
def citation(region):  # 执行本行的状态、计算或校验逻辑。
    return {"doc": region["doc"], "page": region["page"], "bbox": region["bbox"], "kind": region["kind"], "region": region["id"]}  # 执行本行的状态、计算或校验逻辑。
cite = citation(hits[0])  # 执行本行的状态、计算或校验逻辑。
assert cite["region"] == "r2"  # 执行本行的状态、计算或校验逻辑。
assert cite["page"] == 1  # 执行本行的状态、计算或校验逻辑。
assert cite["bbox"] == (0, 130, 400, 260)  # 执行本行的状态、计算或校验逻辑。

## 8. 版本、权限和评测覆盖多模态失败

文档重 OCR、版面模型升级或文件替换都要使 region index 失效。评测分别测 page/region recall、表格单元格准确率、图片事实、OCR 低质量拒答、引用定位和模态路由错误，而不是只测最终回答文本。


In [ ]:
def compatible(region, document_revision):  # 执行本行的状态、计算或校验逻辑。
    return region["doc"] == document_revision  # 执行本行的状态、计算或校验逻辑。
assert compatible(regions[1], "policy-v2")  # 执行本行的状态、计算或校验逻辑。
assert not compatible(regions[1], "policy-v3")  # 执行本行的状态、计算或校验逻辑。
assert all(region["id"] in {"r1", "r2", "r3"} for region in context)  # 执行本行的状态、计算或校验逻辑。

## 面试收束

面试主线是：PDF/图片不是纯文本 chunk；先建立 page-region-bbox-modality 的摄取合同，再讲路由/检索、表格结构、低置信回退、预算和可点击引用。多模态 RAG 的正确性同时包含答案、字段、页码和视觉区域。
